# Exp07: Memory-matched baselines (Days 4--6 revision)

Compare CAS against simple fixed-memory streaming baselines under the same GM-state budget. The memory unit is the number of stored Gaussian-mixture states. For CAS with `L=10`, the comparable budget is `B=L+1=11` protocol-node GM states, excluding the shared prior.

**Sections**
- A: Imports and style
- B: Lightweight Gaussian-mixture state utilities
- C: Baseline buffers
- D: Run K=1 and K=3 comparisons
- E: Paper figures and JSON summary


---
## A. Imports + style

In [1]:
import os, sys, json, math
import numpy as np
import torch
torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd())
from bridge_cas_corr import (
    GaussianMixture,
    make_daily_gaussians_circle, make_daily_gmm_circle,
    run_cl_loop, compute_age_curves,
)

plt.rcParams.update({
    "font.family":        "serif",
    "font.size":          11,
    "axes.labelsize":     12,
    "axes.titlesize":     13,
    "legend.fontsize":    9,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "lines.linewidth":    1.4,
    "lines.markersize":   4,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.05,
})

FIGS = os.path.join(os.getcwd(), 'figs')
os.makedirs(FIGS, exist_ok=True)

C_BLUE   = "#2166ac"
C_ORANGE = "#e08214"
C_GREEN  = "#1a9641"
C_RED    = "#d73027"
C_PURPLE = "#7570b3"
COLORS = {'CAS': C_BLUE, 'FIFO': C_ORANGE, 'Reservoir': C_GREEN,
          'Log-age': C_PURPLE, 'Greedy-PL': C_RED}

N_DAYS = 100
L_DEF = 10
B_STATES = L_DEF + 1
D = 2
R_DRIFT = 2.0
R_COMP = 0.8
PERIOD = 50
COV_K1 = 0.5
COV_K3 = 0.3



---
## B--C. Lightweight GM-state operations and baseline buffers

The baselines use numpy copies of the GM parameters for speed. This does not change the CAS implementation; it only avoids unnecessary PyTorch overhead in the large baseline evaluation loops.

In [2]:
# ------------------------------------------------------------------
# Lightweight numpy GM-state operations for fast baseline evaluation
# ------------------------------------------------------------------

def gm_to_state(gm):
    return dict(w=gm.weights.detach().cpu().numpy().copy(),
                m=gm.means.detach().cpu().numpy().copy(),
                c=gm.covs.detach().cpu().numpy().copy())


def clone_state(s):
    return dict(w=s['w'].copy(), m=s['m'].copy(), c=s['c'].copy())


def interp_state(sa, sb, alpha):
    a = float(np.clip(alpha, 0.0, 1.0))
    w = (1.0 - a) * sa['w'] + a * sb['w']
    w = w / np.sum(w)
    return dict(w=w, m=(1.0 - a) * sa['m'] + a * sb['m'], c=(1.0 - a) * sa['c'] + a * sb['c'])


def state_moments(s):
    w, m, c = s['w'], s['m'], s['c']
    mu = np.einsum('k,kd->d', w, m)
    within = np.einsum('k,kij->ij', w, c)
    dm = m - mu[None, :]
    between = np.einsum('k,ki,kj->ij', w, dm, dm)
    return mu, within + between


def raw_mismatch_states(sa, sb):
    mua, ca = state_moments(sa)
    mub, cb = state_moments(sb)
    return float(np.sum((mua - mub) ** 2) + np.sum((ca - cb) ** 2))


def replay_from_knots(knots, day):
    if not knots:
        raise RuntimeError('empty knot set')
    ks = sorted(knots, key=lambda z: z[0])
    if day <= ks[0][0]:
        return clone_state(ks[0][1])
    if day >= ks[-1][0]:
        return clone_state(ks[-1][1])
    for i, (di, si) in enumerate(ks):
        if day == di:
            return clone_state(si)
        if di > day:
            d0, s0 = ks[i - 1]
            d1, s1 = ks[i]
            alpha = (float(day) - float(d0)) / (float(d1) - float(d0))
            return interp_state(s0, s1, alpha)
    return clone_state(ks[-1][1])


class FIFOBuffer:
    name = 'FIFO'
    def __init__(self, B): self.B, self.items = B, []
    def update(self, day, s):
        self.items.append((int(day), clone_state(s)))
        if len(self.items) > self.B: self.items.pop(0)
    def replay(self, day): return replay_from_knots(self.items, day)
    def n_states(self): return len(self.items)


class ReservoirBuffer:
    name = 'Reservoir'
    def __init__(self, B, seed=0):
        self.B, self.rng = B, np.random.default_rng(seed)
        self.current, self.past, self.n_past_seen = None, [], 0
    def _add_to_reservoir(self, item):
        self.n_past_seen += 1
        cap = max(self.B - 1, 0)
        if cap <= 0: return
        if len(self.past) < cap:
            self.past.append((int(item[0]), clone_state(item[1])))
        else:
            j = self.rng.integers(1, self.n_past_seen + 1)
            if j <= cap:
                self.past[j - 1] = (int(item[0]), clone_state(item[1]))
    def update(self, day, s):
        if self.current is not None: self._add_to_reservoir(self.current)
        self.current = (int(day), clone_state(s))
    def knots(self): return self.past + ([self.current] if self.current is not None else [])
    def replay(self, day): return replay_from_knots(self.knots(), day)
    def n_states(self): return len(self.knots())


class LogAgeBuffer:
    name = 'Log-age'
    def __init__(self, B): self.B, self.current, self.past, self.day = B, None, [], 0
    def update(self, day, s):
        self.day = int(day)
        if self.current is not None: self.past.append((int(self.current[0]), clone_state(self.current[1])))
        self.current = (int(day), clone_state(s))
        cap = max(self.B - 1, 0)
        while len(self.past) > cap: self._drop_one_past()
    def _drop_one_past(self):
        items = sorted(self.past, key=lambda z: z[0])
        if len(items) <= 2:
            items.pop(0)
            self.past = items
            return
        ages = np.array([max(self.day - d, 0) for d, _ in items], dtype=float)
        u = np.log1p(ages)
        best_i, best_score = 1, float('inf')
        for i in range(1, len(items) - 1):
            score = abs(u[i-1] - u[i]) + abs(u[i] - u[i+1])
            if score < best_score: best_i, best_score = i, score
        del items[best_i]
        self.past = items
    def knots(self): return self.past + ([self.current] if self.current is not None else [])
    def replay(self, day): return replay_from_knots(self.knots(), day)
    def n_states(self): return len(self.knots())


class GreedyPLBuffer:
    name = 'Greedy-PL'
    def __init__(self, B): self.B, self.items = B, []
    def update(self, day, s):
        self.items.append((int(day), clone_state(s)))
        self.items = sorted(self.items, key=lambda z: z[0])
        while len(self.items) > self.B: self._drop_min_local_error()
    def _drop_min_local_error(self):
        if len(self.items) <= 2:
            self.items.pop(0); return
        best_i, best_err = 1, float('inf')
        for i in range(1, len(self.items) - 1):
            day_i, s_i = self.items[i]
            d0, s0 = self.items[i - 1]
            d1, s1 = self.items[i + 1]
            pred = interp_state(s0, s1, (day_i - d0) / (d1 - d0))
            err = raw_mismatch_states(s_i, pred)
            if err < best_err: best_i, best_err = i, err
        del self.items[best_i]
    def replay(self, day): return replay_from_knots(self.items, day)
    def n_states(self): return len(self.items)


def make_prior(K, d, cov_scale=1.0):
    return GaussianMixture(weights=torch.ones(K) / K,
                           means=torch.zeros(K, d),
                           covs=cov_scale * torch.eye(d).unsqueeze(0).expand(K, -1, -1).clone())


def run_baseline_matrix(states, prior_state, baseline):
    n_days = len(states)
    base = np.array([raw_mismatch_states(prior_state, states[m]) for m in range(n_days)])
    Fnorm = np.full((n_days + 1, n_days + 1), np.nan)
    mem_states = []
    for n in range(1, n_days + 1):
        baseline.update(n, states[n - 1])
        mem_states.append(baseline.n_states())
        for m in range(1, n + 1):
            rep = baseline.replay(m)
            Fnorm[m, n] = raw_mismatch_states(rep, states[m - 1]) / (base[m - 1] + 1e-15)
    return Fnorm, np.array(mem_states)


def summarize_curve(Fnorm):
    ages, mu, sig, cnt, hl = compute_age_curves(Fnorm, Fnorm.shape[0] - 1)
    return dict(ages=ages, mu=mu, sig=sig, hl=hl)


def half_life_from_curve(ages, curve, theta=0.5):
    for a, v in zip(ages, curve):
        if a > 0 and np.isfinite(v) and v >= theta: return int(a)
    return None


def run_case(case_name, daily_dists, gm_prior, reservoir_seeds=range(5)):
    print(f'\n=== {case_name} ===', flush=True)
    states = [gm_to_state(gm) for gm in daily_dists]
    prior_state = gm_to_state(gm_prior)
    results = {}
    _, _, Fcas, _ = run_cl_loop(daily_dists, gm_prior, L=L_DEF, verbose_every=0)
    results['CAS'] = summarize_curve(Fcas); results['CAS']['states'] = B_STATES
    print(f"  CAS: a1/2={results['CAS']['hl']}, states={B_STATES}", flush=True)
    for cls in [FIFOBuffer, LogAgeBuffer, GreedyPLBuffer]:
        b = cls(B_STATES)
        F, st = run_baseline_matrix(states, prior_state, b)
        results[b.name] = summarize_curve(F); results[b.name]['states'] = int(np.max(st))
        print(f"  {b.name}: a1/2={results[b.name]['hl']}, states={int(np.max(st))}", flush=True)
    curves, hls, stmax = [], [], []
    for seed in reservoir_seeds:
        b = ReservoirBuffer(B_STATES, seed=seed)
        F, st = run_baseline_matrix(states, prior_state, b)
        r = summarize_curve(F)
        curves.append(r['mu']); hls.append(r['hl'] if r['hl'] is not None else np.nan); stmax.append(int(np.max(st)))
    curves = np.vstack(curves)
    ages = np.arange(curves.shape[1])
    mu, sig = np.nanmean(curves, axis=0), np.nanstd(curves, axis=0)
    hl = half_life_from_curve(ages, mu)
    results['Reservoir'] = dict(ages=ages, mu=mu, sig=sig, hl=hl,
                                seed_half_lives=hls, states=int(np.max(stmax)))
    print(f"  Reservoir: a1/2={hl}, states={int(np.max(stmax))}, seeds={len(list(reservoir_seeds))}", flush=True)
    return results


def plot_case(results, title, filename):
    fig, ax = plt.subplots(figsize=(5.6, 3.8))
    for name in ['CAS', 'FIFO', 'Reservoir', 'Log-age', 'Greedy-PL']:
        r = results[name]; ages, mu = r['ages'], r['mu']
        mask = np.isfinite(mu) & (ages > 0)
        hl_txt = str(r['hl']) if r['hl'] is not None else f'>{N_DAYS}'
        lbl = f"{name} ($a_{{1/2}}={hl_txt}$)"
        if name == 'Reservoir':
            ax.fill_between(ages[mask], np.maximum(mu[mask] - r['sig'][mask], 0), mu[mask] + r['sig'][mask],
                            color=COLORS[name], alpha=0.12, lw=0)
        ax.plot(ages[mask], mu[mask], 'o-', ms=2, color=COLORS[name], label=lbl)
    ax.axhline(0.5, ls='--', color='gray', lw=0.6)
    ax.axhline(1.0, ls=':', color='gray', lw=0.4, alpha=0.6)
    ax.set_xlabel('Age $a=n-m$'); ax.set_ylabel(r'Normalised moment error $\bar F(a)$')
    ax.set_title(title); ax.set_xlim(0, N_DAYS); ax.set_ylim(-0.03, 1.55)
    ax.legend(fontsize=7, loc='upper left')
    fig.savefig(os.path.join(FIGS, filename + '.pdf'))
    fig.savefig(os.path.join(FIGS, filename + '.png'))
    plt.close(fig)


def plot_combined(res_k1, res_k3):
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 3.0))
    for ax, results, title in [(axes[0], res_k1, '(a) $K=1$ stream'), (axes[1], res_k3, '(b) $K=3$ stream')]:
        for name in ['CAS', 'FIFO', 'Reservoir', 'Log-age', 'Greedy-PL']:
            r = results[name]; ages, mu = r['ages'], r['mu']
            mask = np.isfinite(mu) & (ages > 0)
            ax.plot(ages[mask], mu[mask], 'o-', ms=1.8, color=COLORS[name], label=name)
        ax.axhline(0.5, ls='--', color='gray', lw=0.5)
        ax.axhline(1.0, ls=':', color='gray', lw=0.3, alpha=0.6)
        ax.set_xlabel('Age $a$'); ax.set_ylabel(r'$\bar F(a)$')
        ax.set_title(title, fontsize=10); ax.set_xlim(0, N_DAYS); ax.set_ylim(-0.03, 1.55)
    axes[0].legend(fontsize=6, loc='upper left')
    ax = axes[2]; methods = ['CAS', 'FIFO', 'Reservoir', 'Log-age', 'Greedy-PL']
    x = np.arange(len(methods))
    h1_raw = [res_k1[m]['hl'] for m in methods]
    h3_raw = [res_k3[m]['hl'] for m in methods]
    h1 = [h if h is not None else N_DAYS for h in h1_raw]
    h3 = [h if h is not None else N_DAYS for h in h3_raw]
    ax.bar(x - 0.18, h1, width=0.34, color=C_BLUE, alpha=0.8, label='$K=1$')
    ax.bar(x + 0.18, h3, width=0.34, color=C_RED, alpha=0.75, label='$K=3$')
    ax.set_xticks(x); ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('$a_{1/2}$'); ax.set_title('(c) Half-life summary', fontsize=10); ax.legend(fontsize=7)
    for i, (a, b) in enumerate(zip(h1, h3)):
        lab1 = str(h1_raw[i]) if h1_raw[i] is not None else f'>{N_DAYS}'
        lab3 = str(h3_raw[i]) if h3_raw[i] is not None else f'>{N_DAYS}'
        ax.text(i - 0.18, a + 1.0, lab1, ha='center', va='bottom', fontsize=6, color=C_BLUE, rotation=0)
        ax.text(i + 0.18, b + 4.5 if lab1 == lab3 and a == b else b + 1.0, lab3, ha='center', va='bottom', fontsize=6, color=C_RED, rotation=0)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGS, 'baseline_memory_matched_comparison.pdf'))
    fig.savefig(os.path.join(FIGS, 'baseline_memory_matched_comparison.png'))
    plt.close(fig)


def plot_curves_two_panel(res_k1, res_k3):
    """Manuscript-ready two-panel age-curve comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
    for ax, results, title in [(axes[0], res_k1, '(a) $K=1$ default stream'),
                               (axes[1], res_k3, '(b) $K=3$ default stream')]:
        for name in ['CAS', 'FIFO', 'Reservoir', 'Log-age', 'Greedy-PL']:
            r = results[name]
            ages, mu = r['ages'], r['mu']
            mask = np.isfinite(mu) & (ages > 0)
            hl_txt = str(r['hl']) if r['hl'] is not None else f'>{N_DAYS}'
            ax.plot(ages[mask], mu[mask], 'o-', ms=2, color=COLORS[name],
                    label=f'{name} ({hl_txt})')
        ax.axhline(0.5, ls='--', color='gray', lw=0.6)
        ax.axhline(1.0, ls=':', color='gray', lw=0.4, alpha=0.6)
        ax.set_xlabel('Age $a=n-m$')
        ax.set_ylabel(r'$\bar F(a)$')
        ax.set_title(title, fontsize=11)
        ax.set_xlim(0, N_DAYS)
        ax.set_ylim(-0.03, 1.55)
    axes[0].legend(fontsize=6.5, loc='upper left', framealpha=0.85)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGS, 'baseline_memory_matched_curves.pdf'))
    fig.savefig(os.path.join(FIGS, 'baseline_memory_matched_curves.png'))
    plt.close(fig)


---
## D--E. Run comparisons and save figures

Baselines:

- **FIFO:** stores the most recent `B` daily GM states.
- **Reservoir:** stores the current GM state plus a uniform reservoir sample of `B-1` previous states; curves are averaged over five seeds.
- **Log-age:** online multiresolution buffer that removes knots in dense regions of log-age.
- **Greedy-PL:** online greedy piecewise-linear compression; after appending a new knot, remove the interior knot best predicted by its neighbors.

All baselines replay by piecewise-linear interpolation of their sorted stored knots, with endpoint holding outside the stored range.

In [3]:
# Execute the baseline experiment

torch.manual_seed(0); np.random.seed(0)
daily_k1, _ = make_daily_gaussians_circle(N_DAYS, R=R_DRIFT, period=PERIOD, cov_scale=COV_K1, d=D)
prior_k1 = make_prior(1, D)
daily_k3, _ = make_daily_gmm_circle(N_DAYS, K=3, R=R_DRIFT, r=R_COMP, period=PERIOD, cov_scale=COV_K3, d=D)
prior_k3 = make_prior(3, D)
res_k1 = run_case('K=1 default', daily_k1, prior_k1)
res_k3 = run_case('K=3 default', daily_k3, prior_k3)
plot_case(res_k1, f'Memory-matched baselines: $K=1$, $B={B_STATES}$ GM states', 'baselines_k1_age_curves')
plot_case(res_k3, f'Memory-matched baselines: $K=3$, $B={B_STATES}$ GM states', 'baselines_k3_age_curves')
plot_combined(res_k1, res_k3)
summary = {}
for cname, res in [('K1_default', res_k1), ('K3_default', res_k3)]:
    summary[cname] = {}
    for m, r in res.items():
        summary[cname][m] = {'half_life': None if r['hl'] is None else int(r['hl']), 'states': int(r['states'])}
        if m == 'Reservoir':
            summary[cname][m]['seed_half_lives'] = [None if np.isnan(x) else int(x) for x in r['seed_half_lives']]
with open(os.path.join(FIGS, 'days4_6_baseline_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSummary:')
print(json.dumps(summary, indent=2))


=== K=1 default ===
  CAS: a1/2=30, states=11
  FIFO: a1/2=17, states=11
  Log-age: a1/2=51, states=11
  Greedy-PL: a1/2=None, states=11
  Reservoir: a1/2=None, states=11, seeds=5

=== K=3 default ===
  CAS: a1/2=30, states=11
  FIFO: a1/2=16, states=11
  Log-age: a1/2=50, states=11
  Greedy-PL: a1/2=14, states=11
  Reservoir: a1/2=None, states=11, seeds=5

Summary:
{
  "K1_default": {
    "CAS": {
      "half_life": 30,
      "states": 11
    },
    "FIFO": {
      "half_life": 17,
      "states": 11
    },
    "Log-age": {
      "half_life": 51,
      "states": 11
    },
    "Greedy-PL": {
      "half_life": null,
      "states": 11
    },
    "Reservoir": {
      "half_life": null,
      "states": 11,
      "seed_half_lives": [
        null,
        98,
        null,
        null,
        null
      ]
    }
  },
  "K3_default": {
    "CAS": {
      "half_life": 30,
      "states": 11
    },
    "FIFO": {
      "half_life": 16,
      "states": 11
    },
    "Log-age": {
      "half_

The main manuscript figure is `figs/baseline_memory_matched_curves.pdf`. Additional standalone figures and the JSON summary are saved in the same directory.

In [4]:
# ═══════════════════════════════════════════════════════════════════════
#  Manuscript-ready baseline figure:
#  figs/baseline_memory_matched_curves.pdf
# ═══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

for ax, results, title in [
    (axes[0], res_k1, r"(a) $K=1$ default stream"),
    (axes[1], res_k3, r"(b) $K=3$ default stream"),
]:
    for name in ["CAS", "FIFO", "Reservoir", "Log-age", "Greedy-PL"]:
        r = results[name]
        ages, mu = r["ages"], r["mu"]
        mask = np.isfinite(mu) & (ages > 0)

        hl_txt = str(r["hl"]) if r["hl"] is not None else f">{N_DAYS}"
        ax.plot(
            ages[mask],
            mu[mask],
            "o-",
            ms=2,
            color=COLORS[name],
            label=f"{name} ({hl_txt})",
        )

    ax.axhline(0.5, ls="--", color="gray", lw=0.6)
    ax.axhline(1.0, ls=":", color="gray", lw=0.4, alpha=0.6)
    ax.set_xlabel(r"Age $a=n-m$")
    ax.set_ylabel(r"$\bar F(a)$")
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, N_DAYS)
    ax.set_ylim(-0.03, 1.55)

axes[0].legend(fontsize=6.5, loc="upper left", framealpha=0.85)

fig.tight_layout()
fig.savefig(os.path.join(FIGS, "baseline_memory_matched_curves.pdf"))
fig.savefig(os.path.join(FIGS, "baseline_memory_matched_curves.png"))
plt.close(fig)

print("Saved", os.path.join(FIGS, "baseline_memory_matched_curves.pdf"))
print("Saved", os.path.join(FIGS, "baseline_memory_matched_curves.png"))

Saved /Users/chertkov/Dropbox/Apps/Overleaf/ContLearnTempMem/experiments/figs/baseline_memory_matched_curves.pdf
Saved /Users/chertkov/Dropbox/Apps/Overleaf/ContLearnTempMem/experiments/figs/baseline_memory_matched_curves.png
